[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riccardoberta/regressione/blob/main/02_primo_scout_automatico.ipynb)

# Il primo scout automatico

Nel primo notebook abbiamo guardato i dati. Ora costruiamo il primo modello predittivo: una formula semplice che stima il valore di mercato a partire da una sola informazione, l'`overall`.

> **Missione** — Costruire una prima regressione lineare: una retta che usa l'overall per prevedere il valore di mercato.


Riapriamo lo stesso file su cui abbiamo costruito le intuizioni nel notebook precedente: il direttore vuole che il nostro primo modello impari sui dati che già conosciamo bene.


In [ ]:
# Setup: imports e download del dataset (solo la prima volta)
import urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = (
    "https://www.dropbox.com/scl/fi/0l5n46qjwcd5moj3w7d8p/"
    "fifa.zip?rlkey=rcqhagvq5ttlvna5t5r3vn1bm&st=uzplzs5o&dl=1"
)
ZIP_PATH = Path("fifa.zip")
DATA_DIR = Path("fifa_data")

if not ZIP_PATH.exists():
    print("Scarico il dataset...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

if not list(DATA_DIR.rglob("*.csv")):
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)

csv_files = (
    list(DATA_DIR.rglob("players_22.csv"))
    or list(DATA_DIR.rglob("*players*.csv"))
)
raw = pd.read_csv(csv_files[0], low_memory=False)
print(f"Caricato: {raw.shape[0]} giocatori, {raw.shape[1]} colonne")
raw.head()


Stessa pulizia di prima: teniamo solo le colonne utili e tagliamo l'1% di giocatori più costosi.


In [ ]:
# Creiamo una versione didattica piccola e pulita
wanted_columns = [
    "short_name", "age", "overall", "potential", "wage_eur", "value_eur",
    "club_name", "league_name", "player_positions"
]

available = [c for c in wanted_columns if c in raw.columns]
df = raw[available].copy()

# Manteniamo solo i giocatori con valore noto e positivo.
df = df.dropna(subset=["value_eur", "age", "overall", "potential"])
df = df[df["value_eur"] > 0]

# Riduciamo l'effetto dei super-outlier per i primi grafici didattici.
df = df[df["value_eur"] <= df["value_eur"].quantile(0.99)]

# Per leggibilita' useremo spesso il valore in milioni di euro.
df["value_mln_eur"] = df["value_eur"] / 1_000_000
if "wage_eur" in df.columns:
    df["wage_k_eur"] = df["wage_eur"] / 1_000

print("Forma del dataset didattico:", df.shape)
df.head(10)

> **Regressione** — Un problema di regressione consiste nel prevedere un numero. Qui il numero e' il valore di mercato del calciatore. Il modello riceve in ingresso una o piu' caratteristiche e restituisce una previsione numerica.


## Prepariamo input e target

Da scout esperti sappiamo che l'overall era la variabile più correlata col valore. Costruiamo allora il modello più semplice possibile: una formula che, dato un solo indizio (l'overall), restituisce una stima del valore. Per addestrarla dobbiamo prima dirgli quali colonne sono **input** e qual è il **target** da prevedere.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
)

X = df[["overall"]]
y = df["value_mln_eur"]

X.head(), y.head()

> **Input e target** — Nel linguaggio del machine learning, gli input sono le informazioni che forniamo al modello. Il target e' cio' che vogliamo prevedere. In questo caso l'input e' overall, il target e' value_mln_eur.


### Teoria — Regressione lineare semplice

Quando l'input è una sola variabile $x$, la regressione lineare cerca la **retta** di equazione:

$\displaystyle \hat{y} \;=\; w \cdot x + b$

dove:

- $\hat{y}$ (si legge "y-cappello") è la **previsione** del modello.
- $w$ è la **pendenza**: di quanto cresce $\hat{y}$ se $x$ aumenta di $1$.
- $b$ è l'**intercetta**: il valore di $\hat{y}$ quando $x = 0$.

Per un dato giocatore con valore reale $y_i$, l'errore commesso dal modello è il **residuo**:

$\displaystyle r_i \;=\; y_i - \hat{y}_i$

Lo scopo dell'algoritmo è scegliere $w$ e $b$ in modo che i residui siano *complessivamente* piccoli.


## Alleniamo il modello

Adesso lasciamo che l'algoritmo trovi da solo la retta migliore. Quando finisce, leggiamo i due numeri che il nostro scout automatico ha imparato: pendenza e intercetta. Questa è letteralmente la sua *ricetta*.


In [ ]:
model = LinearRegression()
model.fit(X, y)

w = model.coef_[0]
b = model.intercept_
print(f"Formula imparata: valore ≈ {w:.2f} * overall + ({b:.2f})")

> **Parametri imparati** — La retta ha due parametri: pendenza w e intercetta b. All'inizio non sappiamo quali valori usare. L'algoritmo li sceglie cercando la retta che riduce l'errore medio sui dati disponibili.


### Teoria — La funzione di costo

"Errore complessivamente piccolo" deve diventare un numero da minimizzare. La regressione lineare usa l'**errore quadratico medio** (MSE):

$\displaystyle \mathrm{MSE}(w, b) \;=\; \frac{1}{n} \sum_{i=1}^{n} \bigl(y_i - (w \cdot x_i + b)\bigr)^2$

Perché il **quadrato**?

- Così gli errori positivi e negativi non si cancellano fra loro.
- Gli errori grandi pesano molto di più degli errori piccoli (penalità più severa).

L'algoritmo cerca la coppia $(w, b)$ che rende $\mathrm{MSE}$ più piccolo possibile. Per la regressione lineare esiste una **formula chiusa** (il *metodo dei minimi quadrati*) che restituisce subito la soluzione — niente tentativi.


## Visualizziamo la retta dello scout automatico

Per capire cosa ha imparato il modello, disegniamolo in azione: i puntini sono i giocatori reali, la retta è la previsione del nostro scout al variare dell'overall.


In [ ]:
x_line = np.linspace(
    df["overall"].min(), df["overall"].max(), 100
).reshape(-1, 1)
y_line = model.predict(x_line)

plt.figure(figsize=(8,5))
plt.scatter(
    df["overall"], df["value_mln_eur"],
    alpha=0.22, label="Giocatori"
)
plt.plot(x_line, y_line, linewidth=3, label="Modello lineare")
plt.xlabel("Overall rating")
plt.ylabel("Valore [milioni di euro]")
plt.title("Prima regressione: valore previsto da overall")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Dove sbaglia il modello? I residui


Una retta che passa nel mezzo della nuvola sembra ragionevole, ma per giudicarla davvero dobbiamo guardare gli **errori** giocatore per giocatore. Calcoliamoli e disegniamoli.


In [ ]:
# Per ogni giocatore: residuo = valore reale - previsione
residui = y - model.predict(X)

plt.figure(figsize=(8,4))
plt.scatter(df["overall"], residui, alpha=0.25)
plt.axhline(0, color="red", linestyle="--", linewidth=2)
plt.xlabel("Overall rating")
plt.ylabel("Residuo [milioni di euro]")
plt.title("Residui del modello: dove e di quanto sbaglia")
plt.grid(True, alpha=0.3)
plt.show()


### Teoria — Come si leggono i residui

Il grafico dei residui mostra l'errore $r_i = y_i - \hat{y}_i$ in funzione dell'input.

- Se i residui sono **sparsi attorno a zero** senza struttura, il modello sta lavorando bene.
- Se mostrano un **andamento** (curva, ventaglio, gruppi), allora c'è informazione che la retta non riesce a catturare: forse serve un modello più ricco.
- Residui **molto grandi** isolati segnalano giocatori "particolari" (outlier) che meriterebbero un'analisi a parte.


---

> **Domanda** — Secondo voi la retta sottostima o sovrastima i campioni? E cosa succede ai giocatori fortissimi, con overall molto alto?


## Misuriamo l'errore

*In generale* il modello quanto sbaglia? Riassumiamo la sua prestazione in tre numeri sintetici. Sono le tre misure standard che useremo per confrontare lo scout con eventuali rivali nei prossimi episodi.


In [ ]:
y_pred = model.predict(X)
mae = mean_absolute_error(y, y_pred)
rmse = mean_squared_error(y, y_pred) ** 0.5
r2 = r2_score(y, y_pred)

print(f"Errore medio assoluto: {mae:.2f} milioni di euro")
print(f"RMSE: {rmse:.2f} milioni di euro")
print(f"R^2: {r2:.3f}")

> **Errore medio assoluto** — Se l'errore medio assoluto vale, per esempio, 3 milioni, significa che in media le previsioni sbagliano di circa 3 milioni di euro. Questa misura e' intuitiva perche' ha la stessa unita' di misura del target.


### Teoria — Tre metriche per giudicare un modello

**MAE** (errore medio assoluto): media dei moduli dei residui. Stessa unità di misura del target.

$\displaystyle \mathrm{MAE} \;=\; \frac{1}{n} \sum_{i=1}^{n} \bigl|\, y_i - \hat{y}_i \,\bigr|$

**RMSE** (radice dell'errore quadratico medio): più severo del MAE perché penalizza di più gli errori grandi.

$\displaystyle \mathrm{RMSE} \;=\; \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$

**$R^2$** (coefficiente di determinazione): la frazione della variabilità del target spiegata dal modello.

$\displaystyle R^2 \;=\; 1 \;-\; \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{n} (y_i - \bar{y})^2}$

- $R^2 = 1$: previsioni perfette.
- $R^2 = 0$: il modello non fa meglio che indovinare sempre la media.
- $R^2 < 0$: il modello fa *peggio* della media (succede davvero, sui dati nuovi).


## Facciamo scouting su alcuni giocatori

Ultima prova: estraiamo dieci giocatori a caso e confrontiamo la previsione del nostro scout col loro vero valore di mercato. Quanto si avvicina, caso per caso?


In [ ]:
sample = df.sample(10, random_state=7).copy()
sample["predicted_value_mln_eur"] = model.predict(sample[["overall"]])
sample["error_mln_eur"] = (
    sample["predicted_value_mln_eur"] - sample["value_mln_eur"]
)
sample[[
    "short_name", "age", "overall", "potential",
    "value_mln_eur", "predicted_value_mln_eur", "error_mln_eur"
]].round(2)

---

> **Cosa dovremmo aver capito** — Abbiamo costruito un primo modello molto semplice. Funziona meglio del tirare a caso, ma e' limitato: uno scout vero non guarda solo l'overall. Nel prossimo notebook aggiungeremo altri indizi.
